<a href="https://colab.research.google.com/github/grimald-i/calculo-numerico/blob/main/atividade_gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Atividade Animação de Eliminação de Gauss
Mariana Froes Grimaldi - 187574


Vinicius Cruz Cuervo - 185337

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.animation as animation

# ==============================================================================
# 1. DEFINIÇÃO DO SISTEMA LINEAR E PONTOS
# ==============================================================================
# Sistema Linear de Exemplo:
#  2x +  y -  z =   8   (Plano E1)
# -3x -  y + 2z = -11   (Plano E2)
# -2x +  y + 2z =  -3   (Plano E3)
#
# Solução: x = 2, y = 3, z = -1

A_orig = np.array([
    [2.0,  1.0, -1.0],
    [-3.0, -1.0,  2.0],
    [-2.0,  1.0,  2.0]
])
b_orig = np.array([8.0, -11.0, -3.0])

# ==============================================================================
# 2. CONSTRUÇÃO DAS ETAPAS (ELIMINAÇÃO + GEOMETRIA 3D + SUBSTITUIÇÃO REGRESSIVA)
# ==============================================================================
frames_data = [
    # Frame 0: Estado Inicial
    {
        'matrix': np.column_stack((A_orig, b_orig)),
        'title': '1. Estado Inicial do Sistema [A | b]',
        'pivot': None, 'target': None,
        'desc': '3 planos em posição geral no espaço 3D, cruzando-se no ponto P(2, 3, -1).',
        'planes': [
            lambda X, Y: 2*X + Y - 8,          # E1: z = 2x + y - 8
            lambda X, Y: 1.5*X + 0.5*Y - 5.5,  # E2: z = 1.5x + 0.5y - 5.5
            lambda X, Y: X - 0.5*Y - 1.5       # E3: z = x - 0.5y - 1.5
        ],
        'plane_labels': ['E1: 2x + y - z = 8', 'E2: -3x - y + 2z = -11', 'E3: -2x + y + 2z = -3'],
        'status_text': 'Sistema original pronto para a Eliminação de Gauss.'
    },
    # Frame 1: Escolha do Pivô 1 (L1, C1)
    {
        'matrix': np.column_stack((A_orig, b_orig)),
        'title': '2. Escolha do Pivô a11 = 2',
        'pivot': (0, 0), 'target': None,
        'desc': 'Pivô selecionado na Linha 1, Coluna 1.',
        'planes': [
            lambda X, Y: 2*X + Y - 8,
            lambda X, Y: 1.5*X + 0.5*Y - 5.5,
            lambda X, Y: X - 0.5*Y - 1.5
        ],
        'plane_labels': ['E1: 2x + y - z = 8', 'E2: -3x - y + 2z = -11', 'E3: -2x + y + 2z = -3'],
        'status_text': 'Pivô = 2.0 (Linha 1)'
    },
    # Frame 2: Eliminação do elemento a21 (-3)
    {
        'matrix': np.array([[2., 1., -1., 8.], [0., 0.5, 0.5, 1.], [-2., 1., 2., -3.]]),
        'title': '3. Eliminação em L2 (L2 ← L2 + 1.5 L1)',
        'pivot': (0, 0), 'target': (1, 0),
        'desc': "E2' não depende de x (0x + 0.5y + 0.5z = 1).\nO plano E2' rotaciona e fica PARALELO ao eixo X!",
        'planes': [
            lambda X, Y: 2*X + Y - 8,
            lambda X, Y: 2 - Y,               # z = 2 - y (paralelo a X)
            lambda X, Y: X - 0.5*Y - 1.5
        ],
        'plane_labels': ['E1: 2x + y - z = 8', "E2': 0.5y + 0.5z = 1 (|| eixo X)", 'E3: -2x + y + 2z = -3'],
        'status_text': 'Multiplicador m21 = -3/2 = -1.5\nL2 ← L2 - (-1.5) L1'
    },
    # Frame 3: Eliminação do elemento a31 (-2)
    {
        'matrix': np.array([[2., 1., -1., 8.], [0., 0.5, 0.5, 1.], [0., 2., 1., 5.]]),
        'title': '4. Eliminação em L3 (L3 ← L3 + L1)',
        'pivot': (0, 0), 'target': (2, 0),
        'desc': "E3' não depende de x (0x + 2y + z = 5).\nOs planos E2' e E3' ficam ambos PARALELOS ao eixo X!",
        'planes': [
            lambda X, Y: 2*X + Y - 8,
            lambda X, Y: 2 - Y,
            lambda X, Y: 5 - 2*Y              # z = 5 - 2y (paralelo a X)
        ],
        'plane_labels': ['E1: 2x + y - z = 8', "E2': 0.5y + 0.5z = 1 (|| eixo X)", "E3': 2y + z = 5 (|| eixo X)"],
        'status_text': 'Multiplicador m31 = -2/2 = -1.0\nL3 ← L3 - (-1.0) L1'
    },
    # Frame 4: Eliminação do elemento a32 (2) -> Matriz Triangular U
    {
        'matrix': np.array([[2., 1., -1., 8.], [0., 0.5, 0.5, 1.], [0., 0., -1., 1.]]),
        'title': '5. Eliminação em L3 (L3 ← L3 - 4 L2)',
        'pivot': (1, 1), 'target': (2, 1),
        'desc': "E3'' reduzido a -z = 1 (z = -1).\nO plano E3'' fica HORIZONTAL (paralelo ao plano XY)!",
        'planes': [
            lambda X, Y: 2*X + Y - 8,
            lambda X, Y: 2 - Y,
            lambda X, Y: -1 * np.ones_like(X) # z = -1 (plano horizontal)
        ],
        'plane_labels': ['E1: 2x + y - z = 8', "E2': 0.5y + 0.5z = 1 (|| eixo X)", "E3'': z = -1 (Horizontal)"],
        'status_text': 'Multiplicador m32 = 2 / 0.5 = 4.0\nL3 ← L3 - 4 L2'
    },
    # Frame 5: Substituição Regressiva & Resultado Final
    {
        'matrix': np.array([[2., 1., -1., 8.], [0., 0.5, 0.5, 1.], [0., 0., -1., 1.]]),
        'title': '6. Resultado Final: Substituição Regressiva',
        'pivot': None, 'target': None,
        'desc': 'De baixo para cima:\n-z = 1 ⇒ z = -1\n0.5y + 0.5(-1) = 1 ⇒ y = 3\n2x + 3 - (-1) = 8 ⇒ x = 2',
        'planes': [
            lambda X, Y: 2*X + Y - 8,
            lambda X, Y: 2 - Y,
            lambda X, Y: -1 * np.ones_like(X)
        ],
        'plane_labels': ['E1: 2x + y - z = 8', "E2': 0.5y + 0.5z = 1", "E3'': z = -1 (Horizontal)"],
        'status_text': 'SOLUÇÃO FINAL DO SISTEMA:\n  x = 2,  y = 3,  z = -1\n  Ponto P(2, 3, -1)'
    }
]

# ==============================================================================
# 3. RENDERIZAÇÃO E ANIMAÇÃO MATPLOTLIB
# ==============================================================================
fig = plt.figure(figsize=(15, 6.5))

x_vals = np.linspace(-0.5, 4.5, 20)
y_vals = np.linspace(0.5, 5.5, 20)
X, Y = np.meshgrid(x_vals, y_vals)
colors = ['#2563eb', '#f59e0b', '#10b981'] # Azul, Laranja, Verde

def render_frame(frame_idx):
    fig.clf()

    data = frames_data[frame_idx]
    mat, title, pivot, target = data['matrix'], data['title'], data['pivot'], data['target']
    desc, planes_f, labels = data['desc'], data['planes'], data['plane_labels']
    status_text = data['status_text']

    # --------------------------------------------------------------------------
    # PAINEL ESQUERDO: Matriz Aumentada e Explicação do Passo
    # --------------------------------------------------------------------------
    ax_mat = fig.add_subplot(1, 2, 1)
    ax_mat.axis('off')
    ax_mat.set_xlim(0, 1)
    ax_mat.set_ylim(0, 1)

    ax_mat.text(0.5, 0.95, title, fontsize=13, weight='bold', ha='center', color='#0f172a')
    ax_mat.text(0.5, 0.85, desc, fontsize=10.5, ha='center', color='#334155', style='italic', multialignment='center')

    rows, cols = mat.shape
    cell_w, cell_h = 0.17, 0.13
    start_x, start_y = 0.10, 0.70

    col_labels = ["x1", "x2", "x3", "b"]
    for c in range(cols):
        cx = start_x + c * cell_w + (0.04 if c == 3 else 0) + cell_w / 2
        ax_mat.text(cx, start_y + 0.03, col_labels[c], fontsize=11, weight='bold', color='#475569', ha='center')

    for r in range(rows):
        for c in range(cols):
            x = start_x + c * cell_w + (0.04 if c == 3 else 0)
            y = start_y - (r + 1) * cell_h

            bg_color, border_color, txt_color, weight = "#f8fafc", "#cbd5e1", "#0f172a", "normal"

            if pivot == (r, c):
                bg_color, border_color, txt_color, weight = "#dbeafe", "#2563eb", "#1d4ed8", "bold"
            elif target == (r, c):
                bg_color, border_color, txt_color, weight = "#fee2e2", "#dc2626", "#b91c1c", "bold"
            elif pivot and r == pivot[0]:
                bg_color = "#eff6ff"
            elif target and r == target[0]:
                bg_color = "#fef2f2"

            rect = plt.Rectangle((x, y), cell_w, cell_h, facecolor=bg_color, edgecolor=border_color, lw=2)
            ax_mat.add_patch(rect)

            val = mat[r, c]
            val_str = f"{val:.2f}".rstrip('0').rstrip('.') if abs(val - round(val)) > 1e-4 else f"{int(round(val))}"
            ax_mat.text(x + cell_w / 2, y + cell_h / 2, val_str, fontsize=12, weight=weight, color=txt_color, ha='center', va='center')

    sep_x = start_x + 3 * cell_w + 0.02
    ax_mat.plot([sep_x, sep_x], [start_y - 3 * cell_h, start_y], color='#475569', lw=2, ls='--')

    # Caixa com Status / Operação
    box_y = 0.22
    rect_status = plt.Rectangle((0.10, box_y), 0.80, 0.12, facecolor='#f1f5f9', edgecolor='#94a3b8', lw=1.5)
    ax_mat.add_patch(rect_status)
    ax_mat.text(0.50, box_y + 0.06, status_text, fontsize=10, weight='bold', color='#1e293b', ha='center', va='center')

    # Legenda dos planos
    ax_mat.text(0.10, 0.14, "Planos no Espaço 3D:", fontsize=10, weight='bold', color='#1e293b')
    for idx, (lbl, col) in enumerate(zip(labels, colors)):
        ax_mat.plot([0.10, 0.16], [0.09 - idx*0.04, 0.09 - idx*0.04], color=col, lw=4)
        ax_mat.text(0.18, 0.09 - idx*0.04, lbl, fontsize=9, color='#334155', va='center')

    # --------------------------------------------------------------------------
    # PAINEL DIREITO: Geometria dos Planos em 3D
    # --------------------------------------------------------------------------
    ax_3d = fig.add_subplot(1, 2, 2, projection='3d')
    ax_3d.set_title("Geometria dos Planos no Espaço 3D", fontsize=13, weight='bold', pad=15)

    for idx, (f_plane, col) in enumerate(zip(planes_f, colors)):
        Z_val = f_plane(X, Y)
        ax_3d.plot_surface(X, Y, Z_val, alpha=0.45, color=col, edgecolor='none')

    # Ponto de interseção invariante (Solução: x=2, y=3, z=-1)
    is_final = (frame_idx == len(frames_data) - 1)
    point_color = '#dc2626' if not is_final else '#16a34a' # Verde destaque no final
    point_size = 120 if not is_final else 180

    ax_3d.scatter([2], [3], [-1], color=point_color, s=point_size, zorder=10, edgecolor='black', linewidth=1.5)

    label_text = " P(2, 3, -1)" if not is_final else " SOLUÇÃO P(2, 3, -1)"
    ax_3d.text(2, 3, -0.2, label_text, color=point_color, fontsize=11, weight='bold')

    ax_3d.set_xlabel('X (x1)', fontsize=10)
    ax_3d.set_ylabel('Y (x2)', fontsize=10)
    ax_3d.set_zlabel('Z (x3)', fontsize=10)
    ax_3d.set_xlim(-0.5, 4.5)
    ax_3d.set_ylim(0.5, 5.5)
    ax_3d.set_zlim(-5, 5)
    ax_3d.view_init(elev=22, azim=-55)

# 4. Criar e Salvar o GIF
anim = animation.FuncAnimation(fig, render_frame, frames=len(frames_data), interval=3000, repeat=True)
anim.save("gauss_3d_completo.gif", writer='pillow', fps=0.33)
plt.close()

print("GIF animado gerado com sucesso como 'gauss_3d_completo.gif'!")

GIF animado gerado com sucesso como 'gauss_3d_completo.gif'!
